In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from sklearn.metrics import accuracy_score
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
MODEL_NAME="distilbert-base-uncased"
MAX_LENGTH=256
BATCH_SIZE=16
EPOCHS=2
LEARNING_RATE=2e-4
BOTTLENECK_SIZE=64
MAX_TRAIN_SAMPLES = 5000   # Set to None for all 5,000 training reviews
MAX_TEST_SAMPLES = 2000    # Set to None for all 2,000 test reviews

In [4]:
from datasets import load_dataset

# AG News dataset
ag_dataset = load_dataset("fancyzhx/ag_news")
print(ag_dataset)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [5]:
train_dataset = ag_dataset['train'].shuffle(seed=42)
test_dataset = ag_dataset['test'].shuffle(seed=42)

if MAX_TRAIN_SAMPLES is not None:
    train_dataset = train_dataset.select(range(min(MAX_TRAIN_SAMPLES, len(train_dataset))))

if MAX_TEST_SAMPLES is not None:
    test_dataset = test_dataset.select(range(min(MAX_TEST_SAMPLES, len(test_dataset))))

print('Training reviews:', len(train_dataset))
print('Testing reviews:', len(test_dataset))
print('Example review:', train_dataset[0]['text'][:400])
print('Example label:', train_dataset[0]['label'])

Training reviews: 5000
Testing reviews: 2000
Example review: Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.
Example label: 0


# tokenize the text using pretrained tokenizer

In [6]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column('label', 'labels')
test_dataset = test_dataset.rename_column('label', 'labels')

columns_to_keep = ['input_ids', 'attention_mask', 'labels']
train_dataset.set_format(type='torch', columns=columns_to_keep)
test_dataset.set_format(type='torch', columns=columns_to_keep)

print('Token IDs:', train_dataset[0]['input_ids'][:20])


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Token IDs: tensor([  101,  7269, 11498,  2135,  6924,  2011,  9326,  4559, 10134,  2031,
         2716,  2116,  4865,  1998,  3655,  1999,  7269,  2000,  1037,  9190])


# create dataloader

In [7]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)#padding to short sentence

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

batch = next(iter(train_loader))
print({key: value.shape for key, value in batch.items()})

{'labels': torch.Size([16]), 'input_ids': torch.Size([16, 94]), 'attention_mask': torch.Size([16, 94])}


# Define the Adapter

In [8]:
class Adapter(nn.Module):
    def __init__(self, hidden_size, bottleneck_size=64):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck_size)
        self.activation = nn.ReLU()
        self.up = nn.Linear(bottleneck_size, hidden_size)

        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        adapter_output = self.down(x)
        adapter_output = self.activation(adapter_output)
        adapter_output = self.up(adapter_output)
        return x + adapter_output

# Wrap a pretrained Transformer block

Each wrapper runs the original frozen DistilBERT block, then sends its hidden states through a trainable adapter. The wrapper preserves the block's output format.

In [9]:
class BlockWithAdapter(nn.Module):
    def __init__(self, original_block, hidden_size, bottleneck_size):
        super().__init__()
        self.original_block = original_block#768 original pretrained DistilBERT Transformer block.
        self.adapter = Adapter(hidden_size, bottleneck_size)#768 to 64

    def forward(self, *args, **kwargs):#allow to pass different inputs to original Transformer block without needing every argument in advance.
        block_output = self.original_block(*args, **kwargs)#That output is stored in block_output.
        #here input goes to origionalblock it proces it produce output and stored in block_output(eg[1,5,768])
#In this code, the Adapter updates hidden states, not attention weights. The attention mechanism itself is inside the original Transformer block.
        # DistilBERT normally returns a tensor; handle tuple output too.
        # Adapter processes hidden_states The Adapter returns adjusted hidden states.768 → 64 → 768New hidden_states
        if isinstance(block_output, tuple):
            hidden_states = self.adapter(block_output[0])
            return (hidden_states,) + block_output[1:]
#The original Transformer block processes the sentence and produces hidden states. The Adapter receives those hidden states, learns a small adjustment, and passes updated hidden states forward.
        return self.adapter(block_output)#If the original block returns only a tensor, we directly pass that tensor through the Adapter and return the result.

#Does the Adapter receive the original tokens?Not directly. It receives the hidden-state representations produced by the original Transformer block

## 9. Build the real pretrained model with adapters

In [10]:
class distilbertwithadapters(nn.Module):
    def __init__(self, MODEL_NAME='distilbert-base-uncased', BOTTLENECK_SIZE=64, num_labels=4):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

        # freeze all pretrained parameters
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

        hidden_size = self.encoder.config.dim
        old_blocks = self.encoder.transformer.layer

        self.encoder.transformer.layer = nn.ModuleList([
            BlockWithAdapter(block, hidden_size, BOTTLENECK_SIZE)
            for block in old_blocks
        ])

        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

model = distilbertwithadapters(
    MODEL_NAME,
    BOTTLENECK_SIZE,
    num_labels=4
)
print(model)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbertwithadapters(
  (encoder): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x BlockWithAdapter(
          (original_block): TransformerBlock(
            (attention): DistilBertSelfAttention(
              (q_lin): Linear(in_features=768, out_features=768, bias=True)
              (k_lin): Linear(in_features=768, out_features=768, bias=True)
              (v_lin): Linear(in_features=768, out_features=768, bias=True)
              (out_lin): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (ffn): FFN(
     

In [12]:
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_parameters:,}')
print(f'Trainable parameters: {trainable_parameters:,}')
print(f'Trainable percentage: {100 * trainable_parameters / total_parameters:.2f}%')

print('\nTrainable parameter names:')
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name, tuple(parameter.shape))

Total parameters: 66,960,772
Trainable parameters: 597,892
Trainable percentage: 0.89%

Trainable parameter names:
encoder.transformer.layer.0.adapter.down.weight (64, 768)
encoder.transformer.layer.0.adapter.down.bias (64,)
encoder.transformer.layer.0.adapter.up.weight (768, 64)
encoder.transformer.layer.0.adapter.up.bias (768,)
encoder.transformer.layer.1.adapter.down.weight (64, 768)
encoder.transformer.layer.1.adapter.down.bias (64,)
encoder.transformer.layer.1.adapter.up.weight (768, 64)
encoder.transformer.layer.1.adapter.up.bias (768,)
encoder.transformer.layer.2.adapter.down.weight (64, 768)
encoder.transformer.layer.2.adapter.down.bias (64,)
encoder.transformer.layer.2.adapter.up.weight (768, 64)
encoder.transformer.layer.2.adapter.up.bias (768,)
encoder.transformer.layer.3.adapter.down.weight (64, 768)
encoder.transformer.layer.3.adapter.down.bias (64,)
encoder.transformer.layer.3.adapter.up.weight (768, 64)
encoder.transformer.layer.3.adapter.up.bias (768,)
encoder.transform

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE
    )

In [15]:
model = model.to(device)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{EPOCHS} - Training loss: {average_loss:.4f}')

Epoch 1/2 - Training loss: 0.4530
Epoch 2/2 - Training loss: 0.3054


In [16]:
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids, attention_mask)
        predictions = logits.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

accuracy = accuracy_score(all_labels, all_predictions)
print(f'Test accuracy on {len(all_labels)} reviews: {accuracy:.4f}')

Test accuracy on 2000 reviews: 0.8880


In [18]:
ag_news_labels = {
    0: 'World',
    1: 'Sports',
    2: 'Business',
    3: 'Sci/Tech'
}

def predict_news_category(sentence):
    model.eval()

    inputs = tokenizer(
        sentence,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        probabilities = torch.softmax(logits, dim=1)[0]

    predicted_class = probabilities.argmax().item()
    label = ag_news_labels[predicted_class]
    confidence = probabilities[predicted_class].item()

    print('Article:', sentence)
    print('Predicted Category:', label)
    print(f'Confidence: {confidence:.2%}')
    print('Probabilities:', {
        ag_news_labels[i]: round(probabilities[i].item(), 4)
        for i in range(len(ag_news_labels))
    })
    print('-' * 60)


# Example test sentences across the 4 AG News categories
predict_news_category('UN leaders met in Geneva today to discuss the new international peace treaty.')
predict_news_category('The championship team won the final match in overtime after a stunning three-pointer.')
predict_news_category('Wall Street stocks surged on Tuesday following strong quarterly earnings reports from major banks.')
predict_news_category('Researchers unveiled a new open-source AI model that runs efficiently on consumer GPUs.')

Article: UN leaders met in Geneva today to discuss the new international peace treaty.
Predicted Category: World
Confidence: 98.82%
Probabilities: {'World': 0.9882, 'Sports': 0.0044, 'Business': 0.0016, 'Sci/Tech': 0.0059}
------------------------------------------------------------
Article: The championship team won the final match in overtime after a stunning three-pointer.
Predicted Category: Sports
Confidence: 98.81%
Probabilities: {'World': 0.0114, 'Sports': 0.9881, 'Business': 0.0002, 'Sci/Tech': 0.0002}
------------------------------------------------------------
Article: Wall Street stocks surged on Tuesday following strong quarterly earnings reports from major banks.
Predicted Category: Business
Confidence: 89.24%
Probabilities: {'World': 0.0902, 'Sports': 0.0024, 'Business': 0.8924, 'Sci/Tech': 0.015}
------------------------------------------------------------
Article: Researchers unveiled a new open-source AI model that runs efficiently on consumer GPUs.
Predicted Category: